# 00 — CircuitKIT Tiny Aya smoke test

Colab-ready check that CircuitKIT can load `CohereLabs/tiny-aya-global` before you spend the weekend on unlearning.

**Do not run this notebook as a training job on the Mac.** Use Colab, Kaggle, or HF ZeroGPU.

Verified references:
- CircuitKIT docs / Tiny Aya port: https://circuitkit.lexsi.ai/advanced/tiny-aya/
- Load API (PyPI / docs): `import circuitkit as ck; ck.load_model("CohereLabs/tiny-aya-global")`
- Import CircuitKIT **before** `transformer_lens`
- Model is gated: accept https://huggingface.co/CohereLabs/tiny-aya-global and set `HF_TOKEN`
- Tiny Aya support is experimental. If this notebook fails, skip circuit comparison and keep probes + logit lens + AuditKIT.

Suggested cheap discovery task in the CircuitKIT docs is `greater_than` (single-token behavior). Forget-set author names often split under Cohere's 262k vocab — do not start with a multi-token name task for the smoke test.



## 1. Runtime and installs



In [ ]:
import sys
print("Python", sys.version)
if sys.version_info < (3, 12):
    print(
        "WARNING: SafeTune requires Python >= 3.12. This smoke test only needs "
        "circuitkit (Python >= 3.10), but the rest of the weekend stack wants 3.12+."
    )

# Install on Colab only. Do not run this cell as a heavy local Mac install.
import os
from pathlib import Path

# If you uploaded the repo, this finds requirements.txt from the repo root.
for cand in [Path.cwd(), Path.cwd().parent, Path("/content/lexsi-unlearning")]:
    req = cand / "requirements.txt"
    if req.exists():
        print("Using", req)
        %pip install -q circuitkit huggingface_hub transformers accelerate
        break
else:
    %pip install -q circuitkit huggingface_hub transformers accelerate


## 2. Hugging Face login (gated model)



In [ ]:
import os
from huggingface_hub import login, HfFolder

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    from google.colab import userdata  # type: ignore
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if not token and Path(".env").exists():
    # Lightweight .env parse (no python-dotenv required). Never print the value.
    for line in Path(".env").read_text().splitlines():
        if line.startswith("HF_TOKEN=") and line.split("=", 1)[1].strip():
            token = line.split("=", 1)[1].strip().strip('"').strip("'")
            break

if token:
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face (token not printed).")
else:
    # Interactive fallback on Colab.
    from huggingface_hub import notebook_login
    print("No HF_TOKEN found in the environment. Opening notebook_login().")
    print("Accept the license first: https://huggingface.co/CohereLabs/tiny-aya-global")
    notebook_login()


## 3. Load Tiny Aya through CircuitKIT

Documented pattern: `import circuitkit as ck` first, then `ck.load_model(...)`.

If load fails with a license / 401 / 403 error, the gate is not accepted or `HF_TOKEN` is missing — fix that rather than switching models.



In [ ]:
# Import circuitkit before transformer_lens (CircuitKIT docs).
import circuitkit as ck

MODEL_ID = "CohereLabs/tiny-aya-global"

model = ck.load_model(MODEL_ID)
print("Loaded", MODEL_ID)
print("type:", type(model))

# Tiny Aya is documented as 36 layers, d_model 2048:
# https://circuitkit.lexsi.ai/advanced/tiny-aya/
for attr in ("cfg", "config", "n_layers", "n_embd"):
    if hasattr(model, attr):
        print(attr, getattr(model, attr))


## 4. Optional tiny discovery smoke

The CircuitKIT docs recommend a single-token task such as `greater_than` for a first run. Keep `n_examples` small.

TODO: confirm the exact `discover` / `discover_circuit` / `Pipeline.discover` signature in the installed package if the call below errors. Docs:
- https://circuitkit.lexsi.ai/
- https://circuitkit.lexsi.ai/advanced/tiny-aya/



In [ ]:
# Keep this optional and tiny. Skip if load_model was already the smoke you needed.
RUN_DISCOVER = False  # flip to True once load_model succeeds and you have a few GPU minutes

if RUN_DISCOVER:
    try:
        circuit = ck.discover(model, "greater_than", n_examples=16)
        print("discover() returned", type(circuit))
        print(circuit)
    except TypeError as exc:
        print("TODO: ck.discover signature did not match. Error:", exc)
        print("Try ck.discover_circuit(...) or Pipeline.discover; see https://circuitkit.lexsi.ai/")
        try:
            from circuitkit import Pipeline
            pipe = Pipeline(MODEL_ID, task="greater_than")
            pipe.discover(algorithm="eap-ig", sparsity=0.3)
            print("Pipeline.discover finished", pipe)
        except Exception as exc2:
            print("Pipeline fallback also failed:", exc2)
            print("Report this as a limitation and continue with probes + logit lens.")
else:
    print("Skipping discovery (RUN_DISCOVER=False). load_model success is enough for a first smoke test.")


## 5. Optional official Tiny Aya parity flag

CircuitKIT documents an optional parity check gated by `CIRCUITKIT_RUN_TINY_AYA=1`. The exact test entry point was not pinned in the public quickstart, so this cell only sets the env var and leaves a TODO rather than guessing a pytest path.



In [ ]:
import os

# TODO: once you have the CircuitKIT source or docs for the parity test, run it here.
# Documented env flag: CIRCUITKIT_RUN_TINY_AYA=1
# https://circuitkit.lexsi.ai/advanced/tiny-aya/
os.environ["CIRCUITKIT_RUN_TINY_AYA"] = "1"
print("Set CIRCUITKIT_RUN_TINY_AYA=1. Look up the official test command before running anything heavy.")
print("Smoke test notebook finished.")
